# Module 4 companion — Transform with dbt (Colab)

This notebook is the Google Colab twin of the repo's **Module 4 — Transform with dbt** and **Exercise 4 — Add a column through the layers**. If you have `uv` set up locally you should run the workshop on your machine instead; this notebook is for people who want the dbt experience with **zero local setup and no clone of the full repo**.

It runs the repo's dbt *pattern* end to end in miniature: **2 raw sources, 2 cleanup views, 1 joined mart** — the medallion shape of the real `transform/` project at a size you can read in one screen. The real project has 18 models, seeds and 75 tests; we deliberately do not reproduce all of that here. What we do keep, because each one is a hard-won repo constraint, is:

- the **trust filter** (dlt writes rows before marking a load successful, so an interrupted run leaves rows behind),
- the **`main_` schema-prefix trap** (dbt-duckdb prepends `main_` to custom schemas by default),
- per-folder materialization (silver = view, gold = table), and
- the two flags everyone forgets: `CRYPTO_DB_PATH` and `--profiles-dir .`.

Unlike the Module 2 notebook, this one does **not** clone the repo: dbt transforms only tables that already exist, and the repo's `transform/` expects a warehouse dlt already filled. So we pre-land three tiny "dlt already did this" tables ourselves and run a real dbt project against them.

**Runtimes:** the install cell is ~2–3 minutes (dbt-core is a big package); every other cell is seconds.

**Caveat:** this notebook has been dry-run on the maintainer's local machine against the pinned `dbt-core==1.12.3`, `dbt-duckdb==1.11.0` and `duckdb==1.5.5`, but has not yet been executed end-to-end inside Colab itself. If a cell fails, please open an issue on the repo with the full traceback.

In [ ]:
# Install the same dbt stack the repo pins in pyproject.toml.
# Pins are deliberate: dbt-core minor versions change macro/Jinja behaviour,
# and this notebook's claims have to match the workshop's behaviour exactly.
#
# We deliberately do NOT install Airflow or dlt here. dbt needs neither; and
# AGENTS.md forbids installing with the Airflow constraints file because it
# pins pathspec==1.1.1, which dbt-core cannot satisfy (it requires <1.1).
%pip install -q "dbt-core==1.12.3" "dbt-duckdb==1.11.0" "duckdb==1.5.5"

# Note: Colab's default Python is 3.12. The repo's pyproject.toml pins
# `requires-python >=3.11,<3.12` for the *full* stack (Airflow is the
# constraint); both dbt pins declare `requires_python >=3.10`, so the dbt
# half is 3.12-clean. If pip reports conflicts with preinstalled Colab
# packages, restart the runtime once and re-run this cell.

## Concept recap

dbt's two reference functions and the two traps this module teaches:

- **`source()`** — read a raw table dbt does not own (the dlt landing zone). No DAG edge; sources are leaves.
- **`ref()`** — read another model *and* wire the dependency edge that builds the DAG. `mart -> stg_fx` exists because the mart's SQL says `ref('stg_fx')`.
- **Materialization** — how a model lands: `view` (thin, computed on read, no disk) or `table` (materialized). The repo sets this per folder in `dbt_project.yml`: bronze/silver = view, gold = table.
- **The trust filter** — dlt writes rows *before* it marks a load successful, so an interrupted run can leave rows whose `load_id` never reaches `status = 0` in dlt's load log. Selecting straight from a raw table would silently include them; the cleanup views inner-join on completed loads so bad rows never reach gold.
- **The `main_` prefix trap** — dbt-duckdb builds custom schema names as `<target-schema>_<custom>` by default, so `+schema: gold` lands in `main_gold` unless you override the `generate_schema_name` macro. The repo ships exactly that macro; we copy it below.

Workshop reference: [`docs/workshop.md` — Module 4](../docs/workshop.md#module-4--transform-with-dbt).

In [ ]:
# Tour the real repo pattern before we rebuild it in miniature. The repo's house
# style is a why-comment on every non-obvious decision; the trust filter is the
# one this module is really about. These bodies are quoted verbatim from
# transform/models/bronze/ — nothing here executes them.
from IPython.display import Markdown, display

br_completed_loads_sql = """\
-- The set of dlt load ids that finished successfully.
--
-- `_dlt_loads` is pipeline-internal bookkeeping in dlt's `bronze` schema. This
-- model is the single, documented place where the rest of the project reads it,
-- so `status = 0` (dlt's "completed" code) is asserted once rather than being
-- repeated as a magic number across every bronze model.
--
-- Note the column names genuinely differ across the join: `_dlt_loads.load_id`
-- against `<table>._dlt_load_id`.

select
    load_id,
    schema_name,
    inserted_at as load_completed_at
from {{ source('bronze', '_dlt_loads') }}
where status = 0
"""

br_coins_markets_sql = """\
-- Bronze: raw payloads restricted to loads that actually completed.
--
-- Structurally 1:1 with the source — no renaming, casting or business logic,
-- which is silver's job. The one thing this layer adds is trust: dlt writes rows
-- before it marks a load successful, so an interrupted run can leave rows behind
-- whose `load_id` never reaches `status = 0`. Selecting straight from the dlt
-- table would silently include them.

select source.*
from {{ source('bronze', 'coins_markets_raw') }} as source
inner join {{ ref('br_completed_loads') }} as loads
    on loads.load_id = source._dlt_load_id
"""

# Why the repo splits this into TWO views: br_completed_loads asserts
# `status = 0` in exactly one place, so the magic number is not repeated in
# every bronze model. Our miniature stays within a 2-view budget, so it inlines
# the same join into each staging view instead — same effect, smaller footprint.
display(Markdown("```sql\n" + br_completed_loads_sql + "\n```"))
display(Markdown("```sql\n" + br_coins_markets_sql + "\n```"))

In [ ]:
# Land the raw tables. In the real repo dlt + CoinGecko/Frankfurter do this;
# dbt transforms only tables that already exist, so a standalone dbt notebook
# has to play the "dlt already landed this" part itself.
#
# DuckDB is single-writer: we open one connection, do all the writes, and close
# it in the same cell — never let a connection span cell boundaries in a
# notebook (the repo's Airflow pool `duckdb_writer` exists for the same reason).
import duckdb

DB_PATH = "/content/mini.duckdb"

con = duckdb.connect(DB_PATH)
try:
    con.execute("create schema if not exists bronze")

    # dlt's load log. One row per load attempt; status=0 means it completed
    # cleanly, anything else means it was interrupted mid-write.
    con.execute(
        """
        create table bronze.loads (
            load_id     varchar,
            status      integer,
            inserted_at timestamp
        )
        """
    )
    # load-good completed; load-bad was interrupted mid-write.
    con.execute(
        """
        insert into bronze.loads values
            ('load-good', 0, '2026-08-30 02:00'),
            ('load-bad',  9, '2026-08-30 02:05')
        """
    )

    # Source 1: the coins (mirror of dlt's coins_markets_raw, minus the
    # variant-column machinery that Module 2's notebook covers).
    con.execute(
        """
        create table bronze.coins_raw (
            id            varchar,
            symbol        varchar,
            price_usd     double,
            _dlt_load_id  varchar,
            _ingested_at  timestamp
        )
        """
    )
    con.execute(
        """
        insert into bronze.coins_raw values
            ('bitcoin',  'btc',  65000.0, 'load-good', '2026-08-30 02:00'),
            ('ethereum', 'eth',  3400.0,  'load-good', '2026-08-30 02:00'),
            -- row from the interrupted load: must NOT survive into the mart
            ('monero',   'xmr',  230.0,   'load-bad',  '2026-08-30 02:05')
        """
    )

    # Source 2: FX rates (mirror of dlt's fx_rates_raw).
    con.execute(
        """
        create table bronze.fx_raw (
            currency      varchar,
            rate_to_usd   double,
            _dlt_load_id  varchar,
            _ingested_at  timestamp
        )
        """
    )
    con.execute(
        """
        insert into bronze.fx_raw values
            ('EUR', 0.92, 'load-good', '2026-08-30 02:00'),
            ('SGD', 0.74, 'load-good', '2026-08-30 02:00'),
            -- interrupted FX load, same bad load_id
            ('IDR', 0.000064, 'load-bad', '2026-08-30 02:05')
        """
    )

    # Row counts before dbt runs — the bad-load rows are visible here on purpose,
    # so the filter's effect later is not a magic trick.
    for t in ("loads", "coins_raw", "fx_raw"):
        n = con.execute(f"select count(*) from bronze.{t}").fetchone()[0]
        print(f"bronze.{t:<10} {n} rows")
finally:
    con.close()

In [ ]:
# Write the miniature dbt project with pathlib, one write_text per file — the
# same files the repo keeps under transform/, in the same shapes, so what you
# read here transfers directly to the real project.
from pathlib import Path

PROJECT = Path("/content/mini_dbt")

files = {
    "dbt_project.yml": """\
name: mini
version: "1.0.0"
profile: mini
model-paths: ["models"]
macro-paths: ["macros"]

models:
  mini:
    silver:
      +schema: silver
      +materialized: view    # thin filters, not copies — views add no disk usage
    gold:
      +schema: gold
      +materialized: table   # the mart is what analysis tools point at
""",
    "profiles.yml": """\
mini:
  target: dev
  outputs:
    dev:
      type: duckdb
      # read from the environment so the notebook (and any CI) can point
      # the same project at a different warehouse without editing this file
      path: "{{ env_var('CRYPTO_DB_PATH') }}"
      schema: main
      threads: 1            # DuckDB is single-writer; more threads buy nothing
""",
    "models/sources/sources.yml": """\
version: 2
sources:
  - name: bronze
    schema: bronze
    tables:
      # the two data sources (mirror of dlt's coins_markets_raw / fx_rates_raw)
      - name: coins_raw
      - name: fx_raw
      # bookkeeping table dlt writes alongside every load — this is what
      # makes the trust filter below possible
      - name: loads
""",
    "models/silver/stg_coins.sql": """\
-- Trust filter: dlt writes rows before it marks a load successful, so an
-- interrupted run can leave rows whose load_id never reached status = 0.
-- Inner-join on completed loads only: bad rows drop out here, never reach gold.
select source.*
from {{ source('bronze', 'coins_raw') }} as source
inner join {{ source('bronze', 'loads') }} as loads
    on loads.load_id = source._dlt_load_id
   and loads.status = 0
""",
    "models/silver/stg_fx.sql": """\
-- Same trust filter as stg_coins, applied to the FX side. The repo splits
-- this into two views (br_completed_loads + one view per table); we inline
-- the join per view to stay within the 2-view budget.
select source.*
from {{ source('bronze', 'fx_raw') }} as source
inner join {{ source('bronze', 'loads') }} as loads
    on loads.load_id = source._dlt_load_id
   and loads.status = 0
""",
    "models/gold/mart_prices.sql": """\
-- One row per currency: every coin priced in each tracked currency.
-- ref() (not source()) on both sides: staging owns the trust filter, and
-- ref() is what wires the DAG edge silver -> gold.
select
    c.id,
    c.symbol,
    f.currency,
    c.price_usd / f.rate_to_usd as price_in_currency
from {{ ref('stg_coins') }} as c
cross join {{ ref('stg_fx') }} as f
""",
    "macros/generate_schema_name.sql": """\
{# dbt-duckdb prepends the target schema (main_) to custom schema names by
   default, so +schema: gold would land in main_gold. Overriding the macro
   uses the custom name verbatim — the repo does exactly this. #}
{% macro generate_schema_name(custom_schema_name, node) -%}
    {%- if custom_schema_name is none -%}
        {{ target.schema }}
    {%- else -%}
        {{ custom_schema_name | trim }}
    {%- endif -%}
{%- endmacro %}
""",
}

for rel, body in files.items():
    path = PROJECT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(body)
print(f"wrote {len(files)} project files under {PROJECT}")

In [ ]:
# Run the build, from the project root, with the two flags the workshop
# warns about in Module 4:
#
#   CRYPTO_DB_PATH   — profiles.yml reads {{ env_var('CRYPTO_DB_PATH') }} with
#                      no default, so dbt errors out without it. Set it with
#                      the %env magic so the `!dbt` subprocess inherits it.
#   --profiles-dir . — the profile lives in the project directory, not ~/.dbt/.
%cd /content/mini_dbt
%env CRYPTO_DB_PATH=/content/mini.duckdb

!dbt build --profiles-dir .


In [ ]:
# Inspect what dbt built. Read-only connection, try/finally close — same
# discipline as the Module 2 notebook.
import duckdb

con = duckdb.connect("/content/mini.duckdb", read_only=True)
try:
    # What exists now. Note the schemas are silver / gold, NOT main_silver /
    # main_gold — that is the generate_schema_name macro doing its job. Delete
    # the macro file and re-run the build to see the trap fire.
    tables = con.execute(
        """
        select table_schema, table_name, table_type
        from information_schema.tables
        where table_schema in ('silver', 'gold')
        order by table_schema, table_name
        """
    ).fetchall()
    print(f"{'schema':<8} {'name':<12} {'type':<10}")
    for schema, name, kind in tables:
        print(f"{schema:<8} {name:<12} {kind:<10}")

    print()
    print("gold.mart_prices (trust filter applied):")
    for r in con.execute(
        "select id, symbol, currency, price_in_currency "
        "from gold.mart_prices order by id, currency"
    ).fetchall():
        print(" ", r)

    print()
    n_bad = con.execute(
        """
        select count(*) from bronze.coins_raw
        where _dlt_load_id = 'load-bad'
        """
    ).fetchone()[0]
    print(f"bronze still holds {n_bad} row(s) from the interrupted load —")
    print("the staging views dropped them, so they never reached gold.")
finally:
    con.close()


## Exercise 4 — Add a column through the layers

The workshop's Exercise 4 asks you to add a new attribute in the gold layer and verify it propagates. The miniature version, same lesson:

**Goal:** add a `market_cap` column through the layers and verify it reaches the mart populated.

**Steps:**

1. Add a `market_cap` value to `bronze.coins_raw` — either re-run the land cell with a sixth value in each `insert` row (drop `/content/mini.duckdb` first), or `alter table ... add column` + `update` on the existing file.
2. `stg_coins` selects `source.*`, so the new column crosses the view for free — verify with `select * from silver.stg_coins`.
3. Add it to the mart's final select: `c.market_cap / f.rate_to_usd as market_cap_in_currency`.
4. Re-run the build cell (`dbt build --profiles-dir .` — a view picks up new columns on read, a table only on rebuild).

**Done when:** `select * from gold.mart_prices` shows the new column populated for all 4 rows and still excludes `monero`.

<details><summary>Solution</summary>

The mart's select becomes:

```sql
select
    c.id,
    c.symbol,
    f.currency,
    c.price_usd / f.rate_to_usd as price_in_currency,
    c.market_cap / f.rate_to_usd as market_cap_in_currency
from {{ ref('stg_coins') }} as c
cross join {{ ref('stg_fx') }} as f
```

For step 1, the simplest correct route is to add the sixth value to each row of the land cell's `insert into bronze.coins_raw values ...`, delete `/content/mini.duckdb`, and re-run cells 4–6. (Edit-and-rerun is exactly how the workshop treats first-run mistakes too.)

</details>


## Cleanup + next steps

**Colab's runtime is ephemeral.** Everything this notebook made lives under `/content` (`mini.duckdb`, `mini_dbt/`) and is gone when the VM shuts down. Nothing to clean up, nothing to keep — the real warehouse belongs on your machine.

**What this notebook did not cover:**

- **The other 15 models** of the real `transform/` — dims, facts, incremental `delete+insert`, the `DECIMAL(38,18)` overflow macro (`money.sql`), the forward-filled FX spine, seeds. [`docs/ARCHITECTURE.md`](../docs/ARCHITECTURE.md) walks every one.
- **dbt tests** — the repo has 75; a miniature with none would be a bad habit to copy. The `data_tests:` entry in Exercise 4 is the pattern.
- **`dbt docs serve`** — the lineage graph is worth seeing on a project with real width; run it locally per Module 4.

**Local setup pointers** if you want the real thing:

```bash
git clone https://github.com/william-dwe/crypto-tracker
cd crypto-tracker
uv sync                       # one command, one venv, all pins
source .venv/bin/activate
uv run ct setup               # first-time: ingest + dbt build (~4 minutes)
cd transform
export CRYPTO_DB_PATH="$PWD/../data/crypto.duckdb"
dbt build --profiles-dir .    # same two flags as this notebook, real project
dbt docs generate --profiles-dir . && dbt docs serve --port 8081 --profiles-dir .
```

Workshop entry point: [`docs/workshop.md`](../docs/workshop.md). Architecture rationale: [`docs/ARCHITECTURE.md`](../docs/ARCHITECTURE.md).
